In [1]:
!pip install konlpy


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
!apt update

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
155 packages can be upgraded. Run 'apt list --upgradable' to see them.


In [3]:
!apt install -y openjdk-17-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 155 not upgraded.


In [4]:
!pip install pandas numpy


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
from konlpy.tag import Okt
import pandas as pd
okt = Okt()

In [ ]:
import re
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df['target'] = df['rating'].apply(lambda x : 1 if x>0.5 else 0)
df['clean'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )    

okt = Okt()
def kor_tokenizer(text):    
    return [
        word for word, pos in okt.pos(text,stem=True) 
            if pos in ['Noun','Verb','Adjective'] and len(word) >=2        
           ]

# 단어사전 구축
from collections import Counter
vocab = Counter()
for text in df['clean']:
    vocab.update(kor_tokenizer(text))

# 10000개의 단어로만 구성
vocab_size = 10000
word_to_index = {
    word:idx+2 for idx, (word,seq) in enumerate(vocab.most_common(vocab_size))
}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

def word2Sequence(text):
    return [word_to_index.get(word,1) for word in kor_tokenizer(text)]    

index_to_word = {idx : word for word, idx in word_to_index.items()}
# padding
import torch
from torch.nn.utils.rnn import pad_sequence
x1 = word2Sequence(df['clean'][0])
x2 = word2Sequence(df['clean'][1])
x1_tensor = torch.LongTensor(x1)
x2_tensor = torch.LongTensor(x2)
print(x1_tensor,x2_tensor)
pad_sequence([x1_tensor,x2_tensor], batch_first=True, padding_value=0)

MAX_LEN = 50
def padding(texts):
    result = []
    for text in texts:
        text = word2Sequence(text)[:MAX_LEN]
        text = torch.LongTensor(text)
        result.append(text)
    return pad_sequence(result,batch_first=True, padding_value=0)
    
padding(df['clean'])  


In [ ]:
# import pandas as pd
# # https://github.com/skn29teacher/skn29_lecture/blob/main/data_nlp/daum_movie_review.csv
# url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
# pd.read_csv(url)

In [ ]:
X = padding(df['clean'])
y = torch.FloatTensor(df['target'].values)

In [ ]:
X[21]

In [ ]:
!pip uninstall -y numpy scipy scikit-learn

In [ ]:
!pip install numpy==1.26.4
!pip install scipy==1.11.4
!pip install scikit-learn==1.3.2

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42)